# Notebook 06 - Validation Against Manual Excel

**Goal:** Compare automated New Bucket assignments (from `archetype_mapping.csv`)
against the analyst's manual assignments (from `Bucket_map` sheet in the Excel file).

**Join key:** `(Division, Portal, Size, bucket_min)` where `bucket_min = floor(manual_min / 100) * 100`

**Metrics:** % rows correctly assigned, % of qty correctly assigned, per-segment breakdown.


In [1]:
import pandas as pd
import numpy as np
import sys, os
import warnings
warnings.filterwarnings('ignore')

# Robust src path - works whether run manually or via papermill
_src_path = os.path.abspath(os.path.join(os.getcwd(), '..', 'src'))
if _src_path not in sys.path:
    sys.path.insert(0, _src_path)
from config import *

OUT      = OUT_PATH   # comes from config import *
RAW_PATH = f'../{VALIDATION_FILE}'

mapping = pd.read_csv(f'{OUT}archetype_mapping.csv')
print(f'archetype_mapping : {mapping.shape}')
print('Columns:', mapping.columns.tolist())

archetype_mapping : (2142, 9)
Columns: ['Division', 'Portal', 'Size', 'bucket_min', 'bucket_max', 'ASP_bucket', 'New_Bucket', 'archetype_key', 'total_qty']


## 1. Load Ground Truth from Excel (`Bucket_map` sheet)

In [2]:
bm_raw = pd.read_excel(RAW_PATH, sheet_name=VALIDATION_SHEET)
print(f'Bucket_map raw shape : {bm_raw.shape}')
print('Years present        :', sorted(bm_raw['year'].unique()))

# Exclude 2023 (same business rule as our pipeline)
bm = bm_raw[bm_raw['year'] >= 2024].copy()
print(f'After excluding 2023 : {bm.shape}')

# Build bucket_min aligned to 100-unit bins (floor(min/100)*100)
if VALIDATION_JOIN == 'floor_100':
    bm['bucket_min'] = (bm[VALIDATION_MIN_COL] // 100) * 100
elif VALIDATION_JOIN == 'parse_string':
    bm['bucket_min'] = bm[VALIDATION_MIN_COL].str.split('-').str[0].astype(int)

# Deduplicate: one ground-truth New Bucket per (Division, Portal, Size, bucket_min)
# Detect qty column: EC uses 'Total_Pcs', TT uses 'qty'
_qty_col = next((c for c in bm.columns if c.lower() in ('qty', 'total_pcs', 'quantity')), None)
if _qty_col is None:
    print(f'WARNING: No qty column found. Available: {bm.columns.tolist()}')
    bm['_qty'] = 0
    _qty_col = '_qty'
else:
    print(f'Using qty column: {_qty_col!r}')

gt = (bm.groupby(['Division', 'Portal', 'Size', 'bucket_min'])
        .agg(gt_New_Bucket=('New Bucket', 'first'),
             gt_key=(GT_KEY_COL, 'first'),
             gt_qty=(_qty_col, 'sum'))
        .reset_index())

print(f'Ground truth unique (seg, bucket_min) rows: {len(gt)}')
print(gt.head(5).to_string(index=False))

Bucket_map raw shape : (8297, 14)
Years present        : [np.int64(2023), np.int64(2024), np.int64(2025)]
After excluding 2023 : (5725, 14)
Using qty column: 'qty'
Ground truth unique (seg, bucket_min) rows: 574
Division  Portal   Size  bucket_min  gt_New_Bucket       gt_key  gt_qty
      BP       1 Single           0              1 BPTT11Single  144049
      BP       1 Single         500              2 BPTT12Single  167249
      BP       1 Single         600              3 BPTT13Single  130747
      BP       1 Single         700              4 BPTT14Single  104946
      BP       1 Single         800              4 BPTT14Single   90213


## 2. Join Automated vs Ground Truth

In [3]:
# Our mapping: Division, Portal, Size, bucket_min, New_Bucket, archetype_key, total_qty
our = mapping[['Division','Portal','Size','bucket_min','New_Bucket','archetype_key','total_qty']].copy()

merged = our.merge(gt[['Division','Portal','Size','bucket_min',
                        'gt_New_Bucket','gt_key','gt_qty']],
                   on=['Division','Portal','Size','bucket_min'],
                   how='left')

n_unmatched = merged['gt_New_Bucket'].isna().sum()
print(f'Rows in our mapping          : {len(our)}')
print(f'Rows matched to ground truth : {len(merged) - n_unmatched}')
print(f'Rows with no GT match        : {n_unmatched}')
if n_unmatched:
    print('  (These are extrapolated buckets beyond GT range - expected)')

Rows in our mapping          : 2142
Rows matched to ground truth : 568
Rows with no GT match        : 1574
  (These are extrapolated buckets beyond GT range - expected)


## 3. Compute Match Metrics

In [4]:
# Only evaluate rows that have a ground truth match
eval_df = merged.dropna(subset=['gt_New_Bucket']).copy()
eval_df['gt_New_Bucket'] = eval_df['gt_New_Bucket'].astype(int)
eval_df['match'] = eval_df['New_Bucket'] == eval_df['gt_New_Bucket']

n_total  = len(eval_df)
n_match  = eval_df['match'].sum()
qty_total = eval_df['total_qty'].sum()
qty_match = eval_df.loc[eval_df['match'], 'total_qty'].sum()

print('=== OVERALL VALIDATION RESULTS ===')
print(f'Rows evaluated       : {n_total}')
print(f'Rows correct         : {n_match}  ({n_match/n_total:.1%})')
print(f'Qty covered (correct): {qty_match:,.0f} / {qty_total:,.0f}  ({qty_match/qty_total:.1%})')

=== OVERALL VALIDATION RESULTS ===
Rows evaluated       : 568
Rows correct         : 118  (20.8%)
Qty covered (correct): 854,076 / 4,214,755  (20.3%)


## 4. Per-Segment Breakdown

In [5]:
seg_res = (eval_df.groupby(['Division','Portal','Size'])
           .apply(lambda g: pd.Series({
               'buckets_evaluated' : len(g),
               'buckets_correct'   : g['match'].sum(),
               'pct_correct'       : f"{g['match'].mean():.1%}",
               'qty_correct_pct'   : f"{g.loc[g['match'],'total_qty'].sum() / g['total_qty'].sum():.1%}"
               if g['total_qty'].sum() > 0 else 'N/A'
           }))
           .reset_index())

print('=== PER-SEGMENT VALIDATION ===')
print(seg_res.to_string(index=False))

=== PER-SEGMENT VALIDATION ===
Division  Portal   Size  buckets_evaluated  buckets_correct pct_correct qty_correct_pct
      BP       1 Single                 20                3       15.0%           25.4%
      BS       1 Single                 10                1       10.0%            0.0%
      DF       1     DF                 25                2        8.0%            6.6%
      DF       1    DFT                 21                5       23.8%           16.1%
      HL       1  CABIN                 50               29       58.0%           37.5%
      HL       1  LARGE                 61               10       16.4%           20.6%
      HL       1 MEDIUM                 59               20       33.9%            1.5%
      HL       1    SO2                 53                9       17.0%           27.7%
      HL       1    SO3                 61                8       13.1%           14.4%
      SL       1  CABIN                 45                1        2.2%            4.7%
 

## 5. Mismatch Detail (for review)

In [6]:
mismatches = eval_df[~eval_df['match']].copy()
mismatches['segment'] = mismatches['Division'] + '_' + mismatches['Portal'].astype(str) + '_' + mismatches['Size']

print(f'Total mismatched bucket-rows: {len(mismatches)}')
print()

if len(mismatches) > 0:
    print('=== MISMATCH DETAIL ===')
    cols = ['segment','bucket_min','New_Bucket','gt_New_Bucket','archetype_key','gt_key','total_qty']
    print(mismatches[cols].sort_values(['segment','bucket_min']).to_string(index=False))
else:
    print('OK Perfect match - zero mismatches!')

Total mismatched bucket-rows: 450

=== MISMATCH DETAIL ===
    segment  bucket_min  New_Bucket  gt_New_Bucket archetype_key        gt_key  total_qty
BP_1_Single         500           4              2  BPTT14Single  BPTT12Single     152707
BP_1_Single         600           4              3  BPTT14Single  BPTT13Single     121192
BP_1_Single         800           5              4  BPTT15Single  BPTT14Single      91707
BP_1_Single         900           5              4  BPTT15Single  BPTT14Single      53848
BP_1_Single        1100           5              6  BPTT15Single  BPTT16Single      79810
BP_1_Single        1200           5              7  BPTT15Single  BPTT17Single      84534
BP_1_Single        1300           5              8  BPTT15Single  BPTT18Single      26835
BP_1_Single        1400           5              8  BPTT15Single  BPTT18Single      38434
BP_1_Single        1500           5              9  BPTT15Single  BPTT19Single      32668
BP_1_Single        2000           6      

## 6. Save Validation Report

In [7]:
# Save full comparison table
try:
    eval_df.to_csv(f'{OUT}06_validation_detail.csv', index=False)
    print(f'OK Saved 06_validation_detail.csv  ({len(eval_df)} rows)')
except PermissionError:
    eval_df.to_csv(f'{OUT}06_validation_detail_new.csv', index=False)
    print(f'NOTE: saved as 06_validation_detail_new.csv — close the original in Excel first')

try:
    seg_res.to_csv(f'{OUT}06_validation_summary.csv', index=False)
    print(f'OK Saved 06_validation_summary.csv ({len(seg_res)} rows)')
except PermissionError:
    seg_res.to_csv(f'{OUT}06_validation_summary_new.csv', index=False)
    print('NOTE: saved as 06_validation_summary_new.csv — close the original in Excel first')
print()
print(f'FINAL: {n_match}/{n_total} bucket-rows correct ({n_match/n_total:.1%})')
print(f'       {qty_match/qty_total:.1%} of total qty correctly assigned')

OK Saved 06_validation_detail.csv  (568 rows)
OK Saved 06_validation_summary.csv (14 rows)

FINAL: 118/568 bucket-rows correct (20.8%)
       20.3% of total qty correctly assigned
